In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

In [16]:
original_labels = [0, 1, 2, 3, 5]
classes = ["airplane", "automobile", "bird", "cat", "dog"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),(0.2470, 0.2435, 0.2616))])

train_base = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=train_transform
)

test_base = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=test_transform
)

Training device: cpu


In [17]:
class FiveClassDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, original_labels):
        self.dataset = dataset
        self.original_labels = original_labels
        self.indices = [i for i, label in enumerate(dataset.targets) if label in original_labels]
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, index):
        original_index = self.indices[index]
        image, original_label = self.dataset[original_index]
        new_label = self.original_labels.index(original_label)
        return image, new_label

train_data = FiveClassDataset(train_base, original_labels)
test_data = FiveClassDataset(test_base, original_labels)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=0)

print("Training images:", len(train_data))
print("Testing images:", len(test_data))

class ImageCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 5))

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = ImageCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001,weight_decay=0.0001)

Training images: 25000
Testing images: 5000


In [18]:
import time
epochs = 15
for epoch in range(epochs):
    start_time = time.time()
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    average_loss = total_loss / len(train_loader)
    train_accuracy = 100 * correct / total
    elapsed = time.time() - start_time
    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Loss: {average_loss:.4f} | "
        f"Train accuracy: {train_accuracy:.2f}% | "
        f"Time: {elapsed:.1f}s"
    )

Epoch 1/15 | Loss: 1.0165 | Train accuracy: 56.53% | Time: 106.2s
Epoch 2/15 | Loss: 0.8167 | Train accuracy: 66.09% | Time: 110.5s
Epoch 3/15 | Loss: 0.7046 | Train accuracy: 71.32% | Time: 110.3s
Epoch 4/15 | Loss: 0.6313 | Train accuracy: 75.15% | Time: 109.0s
Epoch 5/15 | Loss: 0.5810 | Train accuracy: 77.28% | Time: 113.3s
Epoch 6/15 | Loss: 0.5456 | Train accuracy: 78.63% | Time: 110.0s
Epoch 7/15 | Loss: 0.5015 | Train accuracy: 80.60% | Time: 117.6s
Epoch 8/15 | Loss: 0.4864 | Train accuracy: 81.54% | Time: 110.5s
Epoch 9/15 | Loss: 0.4627 | Train accuracy: 82.45% | Time: 110.5s
Epoch 10/15 | Loss: 0.4413 | Train accuracy: 83.12% | Time: 112.7s
Epoch 11/15 | Loss: 0.4260 | Train accuracy: 83.92% | Time: 118.5s
Epoch 12/15 | Loss: 0.4115 | Train accuracy: 84.48% | Time: 109.6s
Epoch 13/15 | Loss: 0.3940 | Train accuracy: 85.07% | Time: 111.2s
Epoch 14/15 | Loss: 0.3882 | Train accuracy: 85.14% | Time: 114.7s
Epoch 15/15 | Loss: 0.3708 | Train accuracy: 86.01% | Time: 111.2s


In [19]:
# TEST THE TRAINED MODEL

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        # Convert original CIFAR-10 labels to labels 0–4
        labels = torch.tensor(
            [original_labels.index(test_base.targets[i])
             for i in test_data.indices[
                 total:total + len(images)
             ]],
            dtype=torch.long,
            device=device
        )

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")
print(f"Correct Predictions: {correct}/{total}")

Test Accuracy: 83.80%
Correct Predictions: 4190/5000


In [21]:
torch.save(model.state_dict(), 'image_recognition_model.pth')